In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.v2 as transforms
from torchvision import datasets
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Usando: {device}')

Usando: cuda


In [2]:
import torch
print(torch.__version__)
import sys
print(sys.version)
print(torch.version.cuda)

2.5.1+cu121
3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
12.1


## Funciones de entrenamiento y validación

La función de test genera un archivo csv que nos permite validar el label predicho y el label correcto.

In [3]:
def train(model, loader, N, optimizer, loss_fn, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in tqdm(loader, desc='  train', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        print("before:", torch.cuda.memory_allocated()/1024**2)
        out = model(x)
        print("after :", torch.cuda.memory_allocated()/1024**2)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += out.argmax(1).eq(y).sum().item()
        print("Allocated:", torch.cuda.memory_allocated()/1024**2, "MB")
        print("Reserved :", torch.cuda.memory_reserved()/1024**2, "MB")
    print(f'  Train | Loss: {total_loss/len(loader):.4f} | Acc: {correct/N:.4f}')
    return correct / N


def validate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out        = model(x)
            total_loss += loss_fn(out, y).item()
            correct    += out.argmax(1).eq(y).sum().item()
            total      += y.size(0)
    print(f'  Val   | Loss: {total_loss/len(loader):.4f} | Acc: {correct/total:.4f}')
    return correct / total

## Definición de transformaciones para data loaders

In [4]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

train_tf = transforms.Compose([
    transforms.Resize((IMG_WIDTH, IMG_HEIGHT)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_WIDTH, IMG_HEIGHT)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

c:\Users\frank\anaconda3\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


## Obtener data sets

In [5]:
# Load data
base_path = os.path.join(os.path.dirname(os.path.abspath('quiz_2.ipynb')), 'data')

# Intel dataset has one extra nested folder level for train/test.
train_path = os.path.join(base_path, 'seg_train', 'seg_train')
val_path   = os.path.join(base_path, 'seg_test', 'seg_test')

train_dataset_full = datasets.ImageFolder(root=train_path, transform=train_tf)
val_dataset_full   = datasets.ImageFolder(root=val_path,   transform=val_tf)

print(f'Clases detectadas: {train_dataset_full.classes}')
print(f'Total train: {len(train_dataset_full):,} imágenes')
print(f'Total val:   {len(val_dataset_full):,} imágenes')

if len(train_dataset_full.classes) <= 1:
    raise ValueError('Ruta de dataset incorrecta: se detecto 1 sola clase.')

Clases detectadas: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Total train: 14,034 imágenes
Total val:   3,000 imágenes


## Dividir validación y test

In [6]:
targets     = np.array(val_dataset_full.targets)
sample_size = 1500

splitter = StratifiedShuffleSplit(n_splits=2, test_size=sample_size, random_state=42)
sample_datasets = []
for _, sample_idx in splitter.split(np.zeros(len(targets)), targets):
    sample_datasets.append(Subset(val_dataset_full, sample_idx))

val_dataset  = sample_datasets[0]
test_dataset = sample_datasets[1]

print(f'Dataset de validación:     {len(val_dataset)} imágenes')
print(f'Dataset de prueba interno: {len(test_dataset)} imágenes')

Dataset de validación:     1500 imágenes
Dataset de prueba interno: 1500 imágenes


## Definir data loaders

In [ ]:
torch.backends.cudnn.benchmark = True
nw = min(4, os.cpu_count())

train_loader = DataLoader(train_dataset_full, batch_size=1, shuffle=True,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)
valid_loader = DataLoader(val_dataset,        batch_size=1, shuffle=False,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset,       batch_size=1, shuffle=False,
                          num_workers=nw, pin_memory=True, persistent_workers=True, prefetch_factor=2)

train_N = len(train_loader.dataset)
valid_N = len(valid_loader.dataset)
test_N  = len(test_loader.dataset)

for images, labels in train_loader:
    print(f'Batch shape: {images.shape}')  # Debe ser [50, 3, 224, 224]
    break

Batch shape: torch.Size([40, 3, 224, 224])


VERIFICAR QUE ID Y LABEL SE OBTIENEN CORRECTAMENTE

In [8]:
def print_first_samples(name, dataset, n=10):
    total = min(n, len(dataset))
    print(f"\n{name} -> primeros {total} samples")

    for i in range(total):
        if isinstance(dataset, Subset):
            base_dataset = dataset.dataset
            base_idx = dataset.indices[i]
            img_path, label_idx = base_dataset.samples[base_idx]
            class_names = getattr(base_dataset, "classes", None)
        else:
            img_path, label_idx = dataset.samples[i]
            class_names = getattr(dataset, "classes", None)

        image_id = os.path.splitext(os.path.basename(img_path))[0]
        label = class_names[label_idx] if class_names is not None else label_idx
        print(f"{i:02d} | id={image_id} | label={label}")


print_first_samples("TRAIN", train_dataset_full, n=10)
print_first_samples("VALID", val_dataset, n=10)
print_first_samples("TEST", test_dataset, n=10)


TRAIN -> primeros 10 samples
00 | id=0 | label=buildings
01 | id=10006 | label=buildings
02 | id=1001 | label=buildings
03 | id=10014 | label=buildings
04 | id=10018 | label=buildings
05 | id=10029 | label=buildings
06 | id=10032 | label=buildings
07 | id=10056 | label=buildings
08 | id=1009 | label=buildings
09 | id=10113 | label=buildings

VALID -> primeros 10 samples
00 | id=23490 | label=forest
01 | id=21740 | label=sea
02 | id=22325 | label=sea
03 | id=20419 | label=sea
04 | id=22220 | label=sea
05 | id=22464 | label=street
06 | id=21356 | label=mountain
07 | id=23586 | label=street
08 | id=22830 | label=street
09 | id=21852 | label=forest

TEST -> primeros 10 samples
00 | id=21229 | label=buildings
01 | id=21793 | label=buildings
02 | id=22089 | label=sea
03 | id=20088 | label=street
04 | id=22172 | label=forest
05 | id=23068 | label=street
06 | id=22138 | label=mountain
07 | id=20147 | label=forest
08 | id=24060 | label=glacier
09 | id=20222 | label=street


# EXPERIMENTO 1
Vamos a entrenar únicamente el último layer del modelo, manteniendo congelados los
demás layers.

In [9]:
#weights = VGG16_Weights.FIXME
weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

N_CLASSES = 6

my_model = nn.Sequential(
    vgg_model.features,
    vgg_model.avgpool,
    nn.Flatten(),
    vgg_model.classifier[0:3],
    nn.Linear(4096, 500),
    nn.ReLU(),
    nn.Linear(500, N_CLASSES)
)
my_model
my_model = my_model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(filter(lambda p: p.requires_grad, my_model.parameters()))

# congelar todo
for p in my_model.parameters():
    p.requires_grad = False

# descongelar solo la ultima capa
for p in my_model[-1].parameters():
    p.requires_grad = True

total     = sum(p.numel() for p in my_model.parameters())
trainable = sum(p.numel() for p in my_model.parameters() if p.requires_grad)

print(f'Parámetros totales:      {total:,}')
print(f'Parámetros entrenables:  {trainable:,}')
print(f'Parámetros congelados:   {total - trainable:,}')

Parámetros totales:      119,530,738
Parámetros entrenables:  3,006
Parámetros congelados:   119,527,732


In [10]:
x, y = next(iter(train_loader))
x = x.to(device)

print("input shape:", x.shape)

print("before:", torch.cuda.memory_allocated()/1024**2)

out = my_model(x)

print("after :", torch.cuda.memory_allocated()/1024**2)

print("output shape:", out.shape)

input shape: torch.Size([40, 3, 224, 224])
before: 479.81884765625


RuntimeError: CUDA error: an illegal memory access was encountered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
epochs = 5

train_accs_exp1 = []
val_accs_exp1   = []
best_val_exp1   = 0.0

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    tr_acc  = train(my_model, train_loader, train_N, optimizer, loss_function, device)
    val_acc = validate(my_model, valid_loader, loss_function, device)
    train_accs_exp1.append(tr_acc)
    val_accs_exp1.append(val_acc)
    if val_acc > best_val_exp1:
        best_val_exp1 = val_acc
        torch.save(my_model.state_dict(), 'best_exp1.pth')

print(f'\nMejor Val Acc Experimento 1: {best_val_exp1:.4f}')
my_model.load_state_dict(torch.load('best_exp1.pth', weights_only=True))

In [ ]:
import torch
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")